# BestShot — Provision Training Environment on Chameleon

This notebook sets up a GPU server on Chameleon for BestShot model training.
It is adapted from the `llm-chi` lab (`2_create_server.ipynb`).

**Before running this notebook:**
- Make sure you have an active lease on KVM@TACC with a GPU node reservation
- Make sure your keypair is registered on Chameleon
- Run this notebook from the Chameleon JupyterHub environment

**What this notebook does:**
1. Connects to your existing lease
2. Brings up the GPU server from a boot volume
3. Sets up security groups and floating IP
4. Installs Docker + NVIDIA container toolkit
5. Clones the BestShot repo
6. Downloads KonIQ-10k dataset
7. Starts MLflow (via Docker Compose)
8. Builds the training Docker container
9. Runs a training job

## Step 1 — Connect to Chameleon project and site

In [ ]:
from chi import server, context, lease, network
import chi, os, time

context.version = "1.0"
context.choose_project()
context.choose_site(default="KVM@TACC")

## Step 2 — Get your existing lease

Replace `bestshot-train-proj19` with the exact name of the lease you created in the Chameleon GUI.
The status should show as **ACTIVE**.

In [ ]:
# Replace with your actual lease name (must end in proj19)
LEASE_NAME = "bestshot-train-proj19"

l = lease.get_lease(LEASE_NAME)
l.show()

## Step 3 — Bring up the server

Creates a 100 GiB boot volume from `CC-Ubuntu24.04-CUDA` and launches the server.
If the server already exists, this cell skips creation.

EfficientNet-B3 is much smaller than LLMs so 100 GiB is plenty (the lab used 200 GiB for BLIP-2).

In [ ]:
username = os.getenv('USER')
server_name = f"bestshot-gpu-proj19"

try:
    s = server.get_server(server_name)
    print(f"Server {server_name} already exists. Skipping create.")
except Exception:
    os_conn = chi.clients.connection()
    cinder_client = chi.clients.cinder()

    images = list(os_conn.image.images(name="CC-Ubuntu24.04-CUDA"))
    image_id = images[0].id

    # 100 GiB is enough for EfficientNet-B3 + KonIQ-10k (~5 GB images)
    boot_vol = cinder_client.volumes.create(
        name=f"boot-vol-bestshot-proj19",
        size=100,
        imageRef=image_id,
    )

    print("Waiting for boot volume to become available...")
    while True:
        boot_vol = cinder_client.volumes.get(boot_vol.id)
        if boot_vol.status == "available":
            break
        if boot_vol.status in ["error", "error_restoring", "error_extending"]:
            raise RuntimeError(f"Boot volume failed with status {boot_vol.status}")
        time.sleep(10)

    bdm = [{
        "boot_index": 0,
        "uuid": boot_vol.id,
        "source_type": "volume",
        "destination_type": "volume",
        "delete_on_termination": True,
    }]

    server_from_vol = os_conn.compute.create_server(
        name=server_name,
        flavor_id=server.get_flavor_id(l.get_reserved_flavors()[0].name),
        block_device_mapping_v2=bdm,
        networks=[{"uuid": os_conn.network.find_network("sharednet1").id}],
    )

    print("Waiting for server to become active...")
    os_conn.compute.wait_for_server(server_from_vol, wait=600)
    s = server.get_server(server_name)
    print(f"Server {server_name} is ready.")

## Step 4 — Security groups

Open SSH (22) and MLflow UI (8000) ports.

In [ ]:
security_groups = [
    {'name': 'allow-ssh',  'port': 22,   'description': 'Enable SSH on port 22'},
    {'name': 'allow-8000', 'port': 8000, 'description': 'Enable MLflow UI on port 8000'},
    {'name': 'allow-9000', 'port': 9000, 'description': 'Enable MinIO on port 9000'},
]

for sg in security_groups:
    secgroup = network.SecurityGroup({
        'name': sg['name'],
        'description': sg['description'],
    })
    secgroup.add_rule(direction='ingress', protocol='tcp', port=sg['port'])
    secgroup.submit(idempotent=True)
    s.add_security_group(sg['name'])

print(f"Security groups applied: {[sg['name'] for sg in security_groups]}")

## Step 5 — Associate floating IP and verify connectivity

In [ ]:
s.associate_floating_ip()

In [ ]:
s.refresh()
s.check_connectivity()

In [ ]:
# Note the floating IP — you'll use it to access MLflow in your browser
s.refresh()
s.show(type="widget")

## Step 6 — Install Docker + NVIDIA container toolkit

Copied directly from the llm-chi lab. Do not skip the `nvidia-ctk` step — it fixes a known cgroup driver issue.

In [ ]:
s.execute("curl -sSL https://get.docker.com | sudo sh")

In [ ]:
s.execute("sudo groupadd -f docker && sudo usermod -aG docker $USER")

In [ ]:
s.execute("""
curl -fsSL https://nvidia.github.io/libnvidia-container/gpgkey \
    | sudo gpg --dearmor -o /usr/share/keyrings/nvidia-container-toolkit-keyring.gpg && \
curl -s -L https://nvidia.github.io/libnvidia-container/stable/deb/nvidia-container-toolkit.list \
    | sed 's#deb https://#deb [signed-by=/usr/share/keyrings/nvidia-container-toolkit-keyring.gpg] https://#g' \
    | sudo tee /etc/apt/sources.list.d/nvidia-container-toolkit.list && \
sudo apt-get -qq update && \
sudo apt-get -qq install -y nvidia-container-toolkit
""")

In [ ]:
# Fix cgroup driver — required for NVIDIA + Docker to work together
s.execute("""
sudo nvidia-ctk runtime configure --runtime=docker && \
sudo systemctl restart docker
""")

In [ ]:
# Verify GPU is visible inside Docker — should show your H100
s.execute("docker run --rm --gpus all nvidia/cuda:12.0-base-ubuntu22.04 nvidia-smi")

## Step 7 — Clone the BestShot repo

In [ ]:
# Replace with your team's actual GitHub repo URL
REPO_URL = "https://github.com/anokhimehta/bestshot"

s.execute(f"git clone {REPO_URL} ~/bestshot")

## Step 8 — Download KonIQ-10k dataset

Downloading directly to the Chameleon instance.
Using 512x384 images (~767 MB) — sufficient for EfficientNet-B3 training.

Source: University of Konstanz VQA group (https://database.mmsp-kn.de/koniq-10k-database.html)

In [ ]:
s.execute("mkdir -p ~/data/koniq10k")

In [ ]:
# Download images (512x384) — ~767 MB, takes a few minutes
s.execute("""
wget -q --show-progress \
    -P ~/data/koniq10k \
    http://datasets.vqa.mmsp-kn.de/archives/koniq10k_512x384.zip
""")

In [ ]:
# Download scores CSV — ~304 KB
s.execute("""
wget -q --show-progress \
    -P ~/data/koniq10k \
    http://datasets.vqa.mmsp-kn.de/archives/koniq10k_scores_and_distributions.zip
""")

In [ ]:
# Unzip both
s.execute("""
cd ~/data/koniq10k && \
unzip -q koniq10k_512x384.zip && \
unzip -q koniq10k_scores_and_distributions.zip
""")

In [ ]:
# Verify — should show 10,073 images and the scores CSV
s.execute("ls ~/data/koniq10k/ && echo '---' && ls ~/data/koniq10k/512x384/ | wc -l")

## Step 9 — Set MLflow tracking URI

MLflow runs on a separate VM provisioned by `mlflow_setup.ipynb`.
Paste the floating IP of that server here before running training.

In [ ]:
# Paste the floating IP from mlflow_setup.ipynb here
MLFLOW_IP = ""  # e.g. "129.114.26.99"

MLFLOW_TRACKING_URI = f"http://{MLFLOW_IP}:8000"
print(f"MLflow tracking URI: {MLFLOW_TRACKING_URI}")

## Step 10 — Build the training container

Builds the Docker image from `training/Dockerfile` and `training/requirements.txt` in the repo.
This takes a few minutes the first time — subsequent builds are faster due to layer caching.

In [ ]:
# Build the training container image
# The build context is training/ so COPY requirements.txt works correctly
s.execute("docker build -t bestshot-train:latest ~/bestshot/training/")

In [ ]:
# Verify the image was built successfully
s.execute("docker images bestshot-train")

## Step 11 — Run a training job

Runs `train.py` inside the container with:
- `--gpus all` so the container can see the GPU
- KonIQ-10k data mounted from the host into `/data/koniq10k` inside the container
- `MLFLOW_TRACKING_URI` pointing at the separate MLflow server from Step 9
- Repo mounted so we can edit `train.py` without rebuilding the image each time

Replace `--config config/baseline.yaml` with whichever config you want to run.

In [ ]:
s.execute(f"""
docker run --rm --gpus all \\
    -v ~/data/koniq10k:/data/koniq10k \\
    -v ~/bestshot:/workspace \\
    -w /workspace \\
    -e MLFLOW_TRACKING_URI={MLFLOW_TRACKING_URI} \\
    bestshot-train:latest \\
    python training/train.py --config training/config/baseline.yaml
""")

## Done!
Environment is fully set up. Once training finishes:

1. Open the MLflow UI at the URL printed in Step 9 to see run
2. Check metrics (MSE, MAE, PLCC) and training time were logged
3. Run additional candidates by changing the config file passed to `--config`